# Vertiefung: Pandas map(), apply(), filter()

**Einordnung im Modul:**  Datenverarbeitung, Datenmanipulation, Datenaufbereitung innerhal einer Datenpipeline

**Kompetenzfokus:** Anwenden von map, filter, apply auf pandas-Objekte. Filterfunktionen `query` und `loc`, sowie die Boolische Indizierung für die Filterung nach inhaltlichen Daten anwenden.

## Lernziele

| Stufe           | Lernziel: Ich kann…                               |
|------------------|----------------------------------------------------|
| 1 Erinnern | typische Anwendungsfälle für .map() und .apply() benennen. |
| 2 Verstehen | erklären, wie .map(), .apply() und .filter() zur Transformation und Filterung von Daten in pandas eingesetzt werden. |
| 2 Verstehen | den Unterschied zwischen .map() und .apply() beschreiben und deren Einsatzgebiete abgrenzen. |
| 2 Verstehen | erläutern, warum query() und iloc[] für komplexe Filterabfragen sinnvoll eingesetzt werden. |
| 3 Anwenden | .map() verwenden, um Werte in einer pandas-Series durch benutzerdefinierte Regeln oder ein Dictionary zu ersetzen. |
| 3 Anwenden | .apply() verwenden, um benutzerdefinierte Funktionen auf Spalten eines pandas-DataFrames anzuwenden. |
| 3 Anwenden | mit .filter() bestimmte Spalten anhand von Namensmustern aus einem DataFrame extrahieren. |
| 3 Anwenden | Filtermethoden wie query() und iloc[] nutzen, um bestimmte Datenpunkte anhand definierter Bedingungen auszuwählen. |
| 4 Analysieren | komplexe Transformationen und Filterungen unter Verwendung von pandas-Methoden kombinieren und deren Effizienz bewerten. |

## Orientierung: Welche Methode wofür?

```text
Eine Spalte (Series) transformieren?
    → .map()

Mehrere Spalten / komplexere Berechnung?
    → .apply()

Spalten nach Namen auswählen?
    → .filter()

Zeilen nach Dateninhalt filtern?
    → Boolean Indexing oder .query()

Zeilen/Spalten nach Position auswählen?
    → .iloc[]
```

### Merksatz

- **`.map()`** = einzelne Werte einer **Series** umwandeln  
- **`.apply()`** = Funktion auf **Series oder Zeilen** anwenden  
- **`.filter()`** = **Spaltennamen** filtern  
- **`query()` / Boolean Indexing** = **Dateninhalt** filtern  
- **`.iloc[]`** = Auswahl nach **Position**

###  Weiterführende Ressourcen

- [Pandas Cheat Sheet](https://pandas.pydata.org/Pandas_Cheat_Sheet.pdf)
- [Pandas Series](https://pandas.pydata.org/pandas-docs/stable/reference/series.html)
- [Pandas DataFrame](https://pandas.pydata.org/pandas-docs/stable/reference/frame.html)

# 1. Setup

In [ ]:
import pandas as pd
from datetime import datetime

s = pd.Series([1, 2, 3])
s

# 2. Pandas `.map()`

`.map()` wird auf **Series** angewendet und transformiert jeden Wert.

Typische Anwendungen:
- Werte durch eine Funktion verändern
- Kategorien über ein Dictionary zuordnen

Wir verwenden im Folgenden zwei Beispiele:
- Flüsse → Kategorien
- Teilnehmerdaten → Alter und Geschlecht

In [ ]:
# Beispiel 1: map() mit lambda
s_quadrat = s.map(lambda x: x**2)
s_quadrat

In [ ]:
# Beispiel 2: map() mit definierter Funktion
def quadrat(x):
    return x**2


s.map(quadrat)

In [ ]:
# Beispiel 3: Dictionary-Mapping mit Flüssen
fluesse = pd.Series(["Nil", "Amazonas", "Jangtse", "Mississippi", "Jenissei", "Mekong"])

fluss_kategorien = {
    "Nil": "Afrika",
    "Amazonas": "Südamerika",
    "Jangtse": "Asien",
    "Mississippi": "Nordamerika",
    "Jenissei": "Asien",
    "Mekong": "Asien",
}

kategorien = fluesse.map(fluss_kategorien)
value_count = kategorien.value_counts()
print(value_count)

print(kategorien)

print(value_count["Asien"])

In [ ]:
# Beispiel 4: Teilnehmerdaten
df_participants = pd.DataFrame(
    {
        "birthday": [
            "1980-03-17",
            "1990-05-23",
            "2000-07-30",
            "1970-09-12",
            "2015-11-05",
            "1995-01-20",
        ],
        "gender": ["m", "f", "f", "m", "f", "m"],
    }
)
df_participants
df_participants.info()

In [ ]:
# Alter berechnen mit map()
def calculate_age(birthdate):
    today = datetime.now()
    return (
        today.year
        - birthdate.year
        - ((today.month, today.day) < (birthdate.month, birthdate.day))
    )


# Umwandlung des birthday-Strings in ein datetime-Objekt
df_participants["birthday"] = pd.to_datetime(df_participants["birthday"])

# Berechnung des Alters und Zuordnung der Geschlechtsbezeichnung
df_participants["age"] = df_participants["birthday"].map(calculate_age)
# Mapping der Geschlechtsbezeichnungen
df_participants["gender_text"] = df_participants["gender"].map(
    {"m": "Male", "f": "Female"}
)

df_participants

### Übung 1 (Anwenden)

Erstelle in `df_participants` eine neue Spalte `age_group`:

- `<20`
- `20-30`
- `30+`

Nutze dafür eine geeignete Transformation auf der Spalte `age`.


### Zwischenfazit zu `.map()`

`.map()` ist besonders stark, wenn:

- **eine einzelne Spalte** transformiert wird
- ein **Dictionary-Mapping** vorliegt
- eine **einfache Funktion** auf jedes Element angewendet werden soll

Für komplexere Berechnungen über mehrere Spalten oder auch bei komplexen Transformationen ist meist `.apply()` geeigneter.
Wenn man mit grössenren Datenmengen arbeitet, sollte man die Effizienz der Methode im Auge behalten. In manchen Fällen kann es sinnvoll sein, auf Vektorisierung oder spezialisierte Funktionen zurückzugreifen, um die Performance zu verbessern. Wir werden das in späteren Einheiten noch vertiefen.


# 3. Pandas `.apply()`

`.apply()` wird verwendet, wenn eine Funktion auf

- eine ganze **Series**
- oder auf **Zeilen/Spalten eines DataFrames**

angewendet werden soll.

Typische Fälle:
- Berechnung aus mehreren Spalten
- komplexere Regeln
- Rückgabe mehrerer Werte


In [ ]:
# Beispiel 1: apply() auf einer Series
# Erzeugen einer Serie
s = pd.Series([1, 2, 3])

s.apply(lambda x: x**2 + 5)

In [ ]:
# Definition der Quadrat-Funktion
def square_and_add_y(x, y):
    return x**2 + y


# Anwenden der Funktion auf die Serie
s_quadrat = s.apply(square_and_add_y, args=(5,))
s_quadrat

In [ ]:
# Beispiel 2: DataFrame mit Schülerpunkten in verschiedenen Fächern
df_punkte = pd.DataFrame(
    {
        "Name": ["Anna", "Ben", "Charlie", "Diana"],
        "Alter": [22, 19, 24, 21],
        "Mathematik": [88, 91, 75, 79],
        "Englisch": [92, 90, 82, 98],
        "Chemie": [79, 95, 78, 99],
    }
)

df_punkte

In [ ]:
# Durchschnitt berechnen mit apply(axis=1)
def calculate_average(row):
    return (row["Mathematik"] + row["Englisch"] + row["Chemie"]) / 3


df_punkte["Durchschnitt"] = df_punkte.apply(calculate_average, axis=1)
df_punkte

In [ ]:
# Alter in Gruppen kategorisieren mit apply() auf einer Series
def categorize_age(age):
    if age < 20:
        return "<20"
    elif age <= 30:
        return "20-30"
    else:
        return "30+"


df_punkte["Alter_Kat"] = df_punkte["Alter"].apply(categorize_age)
df_punkte

In [ ]:
# Beste Punktzahl pro Person
def best_score(row):
    return max(row["Mathematik"], row["Englisch"], row["Chemie"])


df_punkte["Beste_Punktzahl"] = df_punkte.apply(best_score, axis=1)
df_punkte

### Übung 2 (Anwenden)

Ergänze in `df_punkte` die Spalte `Leistungsniveau`:

- Durchschnitt < 80 → `"kritisch"`
- 80 bis 89.9 → `"gut"`
- ab 90 → `"sehr gut"`, ausser bei Schüler:innen, die in einem der Fächer unter 80 haben, dann `"gut"`

Nutze dafür `.apply()`.


# 4. Unterschied `.map()` vs `.apply()`

| Methode | Typischer Einsatz |
|---|---|
| `.map()` | einfache Transformation **einer Series** |
| `.apply()` | komplexere Berechnung |
| `.apply(axis=1)` | Berechnung auf **Zeilenbasis** über mehrere Spalten |
| `.map(Dictionary)` | Kategorien zuordnen / Werte ersetzen |


### Merkhilfe

- **Eine Spalte, einfache Zuordnung?** → `.map()`
- **Mehrere Spalten oder komplexe Logik?** → `.apply()`


# 5. `.filter()` – Spalten nach Namen auswählen

Wichtig: `.filter()` filtert **Spaltennamen** oder **Indexlabels**, **nicht** den Dateninhalt.

Das ist eine häufige Verwechslung.


In [ ]:
# Mini-Beispiel
df_temp = pd.DataFrame(
    {
        "temp_innen": [21.3, 22.1, 20.5],
        "temp_aussen": [11.2, 9.8, 10.5],
        "humidity": [65, 70, 60],
        "temp_max": [23.4, 24.0, 22.7],
    }
)

df_temp

In [ ]:
df_temp[["temp_innen", "temp_max"]]

In [ ]:
# Alle Spalten mit 'temp'
df_temp.filter(like="temp")

In [ ]:
# Alle Spalten, die mit 'temp' beginnen
df_temp.filter(regex="^temp")

In [ ]:
df_temp.filter(items=["temp_innen", "temp_max"])

### Übung 3 (Anwenden)

Extrahiere aus `df_temp`:

1. alle Spalten mit `temp`
2. nur die Spalten `temp_innen` und `humidity`


# 6. Daten nach Inhalt filtern

Wenn nach **Dateninhalt** gefiltert werden soll, verwenden wir typischerweise:

- Boolean Indexing
- `query()`
- `.loc[]`
- `.iloc[]` (positionale Auswahl)


In [ ]:
# Beispiel Boolesche Indizierung: Filtert Daten basierend auf einem booleschen Ausdruck, der auf einer Spalte oder einer Zeile basiert.
# Beispiel DataFrame
df_bewohner = pd.DataFrame(
    {
        "Name": [
            "Anna",
            "Ben",
            "Charlie",
            "Diana",
            "Eve",
            "Frank",
            "Grace",
            "Heidi",
            "Amy",
        ],
        "Alter": [25, 30, 35, 22, 28, 32, 26, 29, 24],
        "Stadt": [
            "Berlin",
            "Hamburg",
            "München",
            "Köln",
            "Frankfurt",
            "Stuttgart",
            "Berlin",
            "Hamburg",
            "München",
        ],
    }
)

df_bewohner

In [ ]:
# Filtern nach einer Bedingung
alter_30_oder_jünger = df_bewohner[df_bewohner["Alter"] <= 30]
alter_30_oder_jünger

In [ ]:
# Filtern mit mehreren Bedingungen
in_berlin_und_jünger_als_30 = df_bewohner.loc[
    (df_bewohner["Stadt"] == "Berlin") & (df_bewohner["Alter"] < 30)
]
in_berlin_und_jünger_als_30

### Übung 4

Filtere alle Personen aus `df_bewohner`, bei denen:

- `Alter <= 30`
- und Name mit "a" beginnt


# 7. `query()` – lesbare Filterformulierung

In [ ]:
# Beispiel `query()`: Filtert Daten basierend auf einem Ausdruck, der in einer speziellen Abfragesprache (SQL-ähnlich) geschrieben ist.
alter_30_oder_jünger = df_bewohner.query("Alter <=30 ")
alter_30_oder_jünger

In [ ]:
# Filtern mit mehreren Bedingungen und spalenten auswählen
in_berlin_und_jünger_als_30 = df_bewohner.query("Stadt == 'Berlin' and Alter < 30")
in_berlin_und_jünger_als_30

### Übung 5

Filtere mit `query()` alle Personen, deren:

- `Alter <= 30`
- und Name mit "a" beginnt


# 8. Wann welche Methode zum Datenfiltern einsetzen?
- Zeilen nach Werten filtern: df.loc[bedingung]
- Viele Bedingungen, gut lesbar: df.query(...)
- Spalten nach Namen auswählen: df[["col1", "col2"]] oder .loc[:, [...]]
- filter() nur für Spezialfälle mit Spalten-/Indexnamen

# 9. Kombination von Transformation und Filterung

Jetzt kombinieren wir mehrere Methoden zu einer kleinen Datenpipeline.

Ziel:
1. Daten filtern
2. neue Information berechnen
3. Ergebnis übersichtlich ausgeben


In [ ]:
df_punkte

In [ ]:
result = (
    df_punkte.query("Alter >=21")
    .assign(
        Leistungsniveau=lambda x: x["Durchschnitt"].apply(
            lambda d: "kritisch" if d < 80 else ("gut" if d < 90 else "sehr gut")
        )
    )
    .filter(items=["Name", "Durchschnitt", "Beste_Punktzahl"])
)

result

### Übung 6 (Analysieren)

Erstelle eine Pipeline auf Basis von `df_punkte`:

1. Filtere nur Personen mit `Alter >= 21`
2. Berechne eine neue Spalte `Bonus`:
   - `5`, wenn `Durchschnitt >= 90`
   - sonst `0`
3. Gib nur die Spalten `Name`, `Durchschnitt`, `Bonus` aus

Nutze eine Kombination aus:
- `query()`
- `.apply()` oder `.map()`
- `.filter()`


## Abschlussreflexion

1. Wann würdest du `.map()` statt `.apply()` verwenden?
2. Warum ist `.filter()` **nicht** dasselbe wie ein Inhaltsfilter?
3. Wann ist `query()` lesbarer als Boolean Indexing?
4. Wann wird eine Pipeline schwer lesbar?


## Lernvideos

Folgende Videos helfen dir bei der Einführung in Pandas map und filter Funktionen:

**Map:**

- ["How to Use map() In Pandas (Python)" - DataDaft](https://www.youtube.com/watch?v=uM4_SY4mXj4)
- ["31 - Pandas DataFrames: Mapping" - Noureddin Sadawi](https://www.youtube.com/watch?v=xkyOEh5ugvc)

**Filter:**

- ["Python Pandas filter() - A Simple Guide" - Finxter](https://www.youtube.com/watch?v=KbNPFKtFxmI)
- ["Query and Filfer Functions (Pandas Tutorials 07)" - Python Tutorials for Digital Humanities](https://www.youtube.com/watch?v=mBZwYUaIRfY)

**Apply:**

- ["Pandas Functions: Three Ways to Use the Apply Function"](https://www.youtube.com/watch?v=DsjvCKxOdgI&ab_channel=M%C4%B1sraTurp)

**Filter nach Dateninhalt:**

- ["Filtering Columns and Rows in Pandas | Python Pandas Tutorials"](https://www.youtube.com/watch?v=kB7FV-ijdqE&ab_channel=AlexTheAnalyst)